In [1]:
# Visualização dos dados de entrada
import pandas as pd
arq = './dados/integras_experimento_summa_novos.parquet'
df = pd.read_parquet(arq)
df[df['seq_documento_acordao']==184283670].iloc[0]

sg_classe                               AREsp
pasta                    ./saidas/<<tipo>>_07
fold                                        7
dt_publicacao                      2023-04-11
sg_ramo_direito                            TR
num_registro                     202203002336
ano                                      2023
seq_documento_acordao               184283670
num_ministro                           1151.0
integra                    #ERRO_CARGA_ITEOR#
index                                     NaN
Name: 5154, dtype: object

In [6]:
# Visualização dos dados de entrada
import pandas as pd
arq = './dados/integras_experimento_summa_novos.parquet'
arq = './dados/teste_integras_experimento_summa.parquet'
df = pd.read_parquet(arq)
#df[df['fold'] == 12]
#df[(df['integra'] == '') & (df['fold'] == 12)]
#df

dfs = df[['seq_documento_acordao','ano','fold']]
dfs.to_csv('./saida/filtro_teste_completo.csv')
#print(df)

# fold 12
dfs = df[(df['integra'] != '') & (df['fold'] == 12)]
dfs = dfs[['seq_documento_acordao','ano','fold']]
dfs.to_csv('./saida/filtro_teste_fold_12.csv')

# fold 11
dfs = df[(df['integra'] != '') & (df['fold'] == 11)]
dfs = dfs[['seq_documento_acordao','ano','fold']]
dfs.to_csv('./saida/filtro_teste_fold_11.csv')

# fold 1 a 10
dfs = df[(df['integra'] != '') & (df['fold'].between(1, 10) )]
dfs = dfs[['seq_documento_acordao','ano','fold']]
dfs.to_csv('./saida/filtro_teste_fold_1a10.csv')

dfs

,seq_documento_acordao,ano,fold
0,147347420,2022,9
1,148041943,2022,6
2,151424778,2022,10
3,151423986,2022,1
4,151424800,2022,7
...,...,...,...
21793,289154252,2025,1
21794,289154087,2025,5
21795,289154861,2025,2
21796,289154813,2025,1


In [10]:
# União dos dataframes e contagem de tokens

import pandas as pd
import json
#arq = './saida/saida_1.5b.parquet'
origem = './dados/integras_experimento_summa_novos.parquet'
resposta = './saida/saida_oa_gpt5.parquet'
df_origem = pd.read_parquet(origem)
df_resposta = pd.read_parquet(resposta)

df_resposta['chave'] = df_resposta['chave'].astype(int)
df_merge = df_origem.merge(df_resposta, left_on='seq_documento_acordao', right_on='chave', how='inner')

#dados = dados[-10:].copy().reset_index()
print(df_merge.iloc[0])
q = len(df_merge)

print('-='*40)

# fold 12
#dfs = df_merge[df_merge['fold'] == 12]
dfs = df_merge[df_merge['fold'] <= 10]
#dfs = dfs[dfs['alvo'] == 'treino']
entrada = 0
saida = 0
print(dfs.iloc[0])

maiores_32k = 0

for i, row in dfs.iterrows():
    dados = row['resumo']
    dados = json.loads(dados)
    #print(row)
    entrada += dados.get('prompt_tokens',0)
    saida += dados.get('completion_tokens',0)
    if dados.get('total_tokens',0) > 32*1024:
        maiores_32k += 1

print('Registros:', len(dfs), 'de', len(df_merge))
print(f'Tokens de entrada:', entrada)    # R$5 
print(f'Tokens de saída:', saida)        # R$20
print(f'Total Custo Sabiá:', (entrada*5 + saida*20) / 1000000)
print(f'Total > 32k:', maiores_32k)

sg_classe                                                            AREsp
pasta                                                 ./saidas/<<tipo>>_01
fold                                                                     1
dt_publicacao                                                   2024-12-17
sg_ramo_direito                                                         TR
num_registro                                                  202301681297
ano                                                                   2024
seq_documento_acordao                                            286006309
num_ministro                                                        1187.0
integra                  ACÓRDÃO\nVistos e relatados estes autos em que...
index                                                                  NaN
chave                                                            286006309
resumo                   {"prompt_tokens": 9824, "completion_tokens": 3...
resposta                 

In [1]:
import pandas as pd
import os
arquivos = ['saida_oa_gpt5.parquet', 'saida_qwen7b.parquet', 'saida_or_235b.parquet', 'saida_sabia4.parquet']

for arq in arquivos:
    arq = os.path.join('./saida', arq)
    df = pd.read_parquet(arq)
    df['tamanho'] = [len(_) for _ in df['resposta']]
    df = df[df['tamanho'] > 10]
    #print(df['resposta'])
    print(f'Arquivo: {arq} com {len(df)} registros')
    #print(df)

print('=' * 50)
print('Saída de um registro')
print(df.iloc[0])
 

Arquivo: ./saida/saida_oa_gpt5.parquet com 22269 registros
Arquivo: ./saida/saida_qwen7b.parquet com 22269 registros
Arquivo: ./saida/saida_or_235b.parquet com 22269 registros
Arquivo: ./saida/saida_sabia4.parquet com 468 registros
Saída de um registro
chave                                               375924114
resumo      {"prompt_tokens": 15225, "completion_tokens": ...
resposta    {\n  "Tipo": "ResAcordao",\n  "Materia": "DIRE...
erro                                                         
tamanho                                                  6823
Name: 0, dtype: object


In [5]:
import json
# União dos dataframes e contagem de tokens

import pandas as pd
import json
#arq = './saida/saida_1.5b.parquet'
origem = './dados/integras_experimento_summa_novos.parquet'
resposta = './saida/saida_oa_gpt5.parquet'
df_origem = pd.read_parquet(origem)
df_resposta = pd.read_parquet(resposta)

df_resposta['chave'] = df_resposta['chave'].astype(int)
df_merge = df_origem.merge(df_resposta, left_on='seq_documento_acordao', right_on='chave', how='inner')

# falha na extração íntegra 184283670
# mais baixa similaridade GPT5 203429360

dados = df_merge[df_merge['seq_documento_acordao']==203429360].iloc[0]
resposta = dados['resposta']
meta = {c:v for c,v in dados.items() if c not in ['resposta','integra']}
print('METADADOS:', meta)
try:
    j = json.loads(resposta)
    print('RESPOSTA:')
    print(j)
except Exception as e:
    print('ERRO:', e)
    print(resposta)
print('ÍNTEGRA:')
print(dados['integra'])

METADADOS: {'sg_classe': 'HC', 'pasta': './saidas/<<tipo>>_09', 'fold': np.int64(9), 'dt_publicacao': '2023-08-17', 'sg_ramo_direito': 'PN', 'num_registro': '202203817335', 'ano': np.int64(2023), 'seq_documento_acordao': np.int64(203429360), 'num_ministro': np.float64(8420.0), 'index': np.float64(nan), 'chave': np.int64(203429360), 'resumo': '{"prompt_tokens": 13360, "completion_tokens": 2908, "total_tokens": 16268, "finish_reason": "stop", "model": "gpt-5-2025-08-07", "tempo": 19.472}', 'erro': ''}
RESPOSTA:
{'Tipo': 'ResAcordao', 'Materia': 'AGRAVO REGIMENTAL NO HABEAS CORPUS. TRÁFICO DE DROGAS. VIOLAÇÃO DE DOMICÍLIO. SUPRESSÃO DE INSTÂNCIA. IMPOSSIBILIDADE DE REVOLVIMENTO FÁTICO-PROBATÓRIO.', 'Dispositivo': 'Ante o exposto, nego provimento ao agravo regimental.', 'DataJulgamento': '2023-08-14', 'Partes': [{'Tipo': 'paciente', 'Nome': 'Alexandre de Sá'}], 'Temas': [{'Ponto': 'Violação de domicílio não apreciada pelo Tribunal a quo (supressão de instância)', 'Preliminar': 'Sim', 'Argu

In [ ]:
# Mostra o último registro com erro
# import json
erros = dados[dados['erro'] > '']
if len(erros) > 0:
    resposta = erros['resposta'].iloc[-1]
    registro = erros['num_registro'].iloc[-1]
    doc = erros['seq_documento_acordao'].iloc[-1]
    try:
        j = json.loads(resposta)
        print(j)
    except Exception as e:
        print('ERRO:', e)
        print(resposta)
else:
    print('nenhum erro encontrado')

In [12]:
# Visualização dos dados de entrada
import pandas as pd
arq = './dados/pecas_exportadas_textos.parquet'
df = pd.read_parquet(arq)
df[df['seq_documento_acordao','dt_publicacao','fold','pasta']]



KeyError: ('seq_documento_acordao', 'dt_publicacao', 'fold', 'pasta')